In [2]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score)

from imblearn.metrics import geometric_mean_score, specificity_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import ADASYN

warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv('/content/bank-additional-full (1).csv', sep=';')
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [6]:
# Drop unnecessary column
df.drop('duration', axis=1, inplace=True)

# Handle missing values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

df.head()

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
1,57,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
2,37,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
3,40,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
4,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False


In [7]:
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [8]:
print("Applying ADASYN Sampling...")

adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_s, y_train)

print("Before Sampling:", y_train.value_counts())
print("After Sampling:", pd.Series(y_train_ad).value_counts())

Applying ADASYN Sampling...
Before Sampling: y
0    29238
1     3712
Name: count, dtype: int64
After Sampling: y
0    29238
1    29184
Name: count, dtype: int64


In [9]:
def get_metrics(y_true, y_pred, y_prob=None):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='weighted'),
        "Recall": recall_score(y_true, y_pred, average='weighted'),
        "Specificity": specificity_score(y_true, y_pred, average='weighted'),
        "F1 Score": f1_score(y_true, y_pred, average='weighted'),
        "ROC-AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else 0.0,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "G-Mean": geometric_mean_score(y_true, y_pred, average='weighted'),
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

In [10]:
models_dict = {

    "Logistic Regression": (
        LogisticRegression(max_iter=500),
        {"C": [0.1, 1]}
    ),

    "Decision Tree": (
        DecisionTreeClassifier(),
        {"max_depth": [5, 10]}
    ),

    "Random Forest": (
        RandomForestClassifier(n_estimators=50),
        {"max_depth": [2, 3]}
    ),

    "GBM": (
        GradientBoostingClassifier(n_estimators=50),
        {"learning_rate": [0.1, 0.8]}
    ),

    "XGBoost": (
        XGBClassifier(n_estimators=50, eval_metric='logloss'),
        {"learning_rate": [0.1, 0.6]}
    ),

    "LightGBM": (
        LGBMClassifier(n_estimators=50, verbose=-1),
        {"learning_rate": [0.1, 0.3]}
    ),

    "CatBoost": (
        CatBoostClassifier(iterations=50, verbose=0),
        {"depth": [3, 4]}
    ),

    "SVM": (
        SVC(probability=True),
        {"C": [0.1, 1], "kernel": ['rbf']}
    ),

    "KNN": (
        KNeighborsClassifier(),
        {"n_neighbors": [2, 5]}
    ),

    "MLP": (
        MLPClassifier(max_iter=200),
        {"hidden_layer_sizes": [(50,), (50, 50)], "activation": ['relu', 'tanh']}
    ),

    "Bagging": (
        BaggingClassifier(n_estimators=10),
        {"max_samples": [0.1, 0.8]}
    ),
}

In [11]:
from sklearn.linear_model import LogisticRegression

base_est = [
    ('rf', RandomForestClassifier(n_estimators=10)),
    ('dt', DecisionTreeClassifier(max_depth=5))
]

models_dict["Stacking"] = (
    StackingClassifier(estimators=base_est, final_estimator=LogisticRegression()),
    {}
)

In [12]:
all_results = []

for name, (model, params) in models_dict.items():

    rs = RandomizedSearchCV(
        model, params,
        n_iter=2,
        cv=2,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train_ad, y_train_ad)

    best_model = rs.best_estimator_

    y_pred = best_model.predict(X_test_s)
    y_prob = best_model.predict_proba(X_test_s)[:, 1] if hasattr(best_model, "predict_proba") else None

    metrics = get_metrics(y_test, y_pred, y_prob)
    metrics["Model"] = name
    all_results.append(metrics)

    print(f"\nModel: {name}")
    print(metrics)


Model: Logistic Regression
{'Accuracy': 0.7869628550619082, 'Precision': 0.8815616075997128, 'Recall': 0.7869628550619082, 'Specificity': np.float64(0.7123569831380068), 'F1 Score': 0.8192772732702832, 'ROC-AUC': np.float64(0.7977554218359356), 'MCC': np.float64(0.36061069626361264), 'G-Mean': np.float64(0.7487312503652919), 'Kappa': np.float64(0.31906587663175945), 'Model': 'Logistic Regression'}

Model: Decision Tree
{'Accuracy': 0.8829813061422676, 'Precision': 0.8786055797959985, 'Recall': 0.8829813061422676, 'Specificity': np.float64(0.4940535775786627), 'F1 Score': 0.8806641681888795, 'ROC-AUC': np.float64(0.7361338536015849), 'MCC': np.float64(0.39227810148902165), 'G-Mean': np.float64(0.6604847259662162), 'Kappa': np.float64(0.3917780773236934), 'Model': 'Decision Tree'}

Model: Random Forest
{'Accuracy': 0.7416848749696529, 'Precision': 0.8778639848481085, 'Recall': 0.7416848749696529, 'Specificity': np.float64(0.7301286539703914), 'F1 Score': 0.7858186779565245, 'ROC-AUC': n

In [13]:
all_results = []

for name, (model, params) in models_dict.items():

    rs = RandomizedSearchCV(
        model, params,
        n_iter=2,
        cv=2,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train_ad, y_train_ad)

    best_model = rs.best_estimator_

    y_pred = best_model.predict(X_test_s)
    y_prob = best_model.predict_proba(X_test_s)[:, 1] if hasattr(best_model, "predict_proba") else None

    metrics = get_metrics(y_test, y_pred, y_prob)
    metrics["Model"] = name
    all_results.append(metrics)

    print(f"\nModel: {name}")
    print(metrics)


Model: Logistic Regression
{'Accuracy': 0.7869628550619082, 'Precision': 0.8815616075997128, 'Recall': 0.7869628550619082, 'Specificity': np.float64(0.7123569831380068), 'F1 Score': 0.8192772732702832, 'ROC-AUC': np.float64(0.7977554218359356), 'MCC': np.float64(0.36061069626361264), 'G-Mean': np.float64(0.7487312503652919), 'Kappa': np.float64(0.31906587663175945), 'Model': 'Logistic Regression'}

Model: Decision Tree
{'Accuracy': 0.8833454722019908, 'Precision': 0.8788224931336914, 'Recall': 0.8833454722019908, 'Specificity': np.float64(0.4940998082357657), 'F1 Score': 0.8809442434836795, 'ROC-AUC': np.float64(0.7363889511297702), 'MCC': np.float64(0.3933217550353604), 'G-Mean': np.float64(0.660651820871581), 'Kappa': np.float64(0.3927804802919298), 'Model': 'Decision Tree'}

Model: Random Forest
{'Accuracy': 0.7442340373877154, 'Precision': 0.8779314383623571, 'Recall': 0.7442340373877154, 'Specificity': np.float64(0.7285706939675372), 'F1 Score': 0.7877150051815891, 'ROC-AUC': np.

In [23]:


columns = [
    "Accuracy", "Precision", "Recall",
    "Specificity", "F1 Score", "ROC-AUC",
    "MCC", "G-Mean", "Kappa", "Model"
]


final_df = pd.DataFrame(all_results, columns=columns)



final_df = final_df[
    ["Model", "Accuracy", "Precision", "Recall",
     "Specificity", "F1 Score", "ROC-AUC",
     "MCC", "G-Mean", "Kappa"]
]



final_df = final_df.round(4)

final_df.to_excel("output.xlsx", index=False)
